In [0]:
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    rank, dense_rank, row_number, ntile,
    percent_rank, cume_dist
)

data = [
    ("Ravi",   "Engineering", "Pune",      55000, 28),
    ("Priya",  "HR",          "Mumbai",    42000, 32),
    ("Arjun",  "Engineering", "Delhi",     72000, 26),
    ("Sneha",  "Finance",     "Pune",      61000, 30),
    ("Rohit",  "Engineering", "Mumbai",    80000, 35),
    ("Meera",  "HR",          "Bangalore", 39000, 27),
    ("Karan",  "Finance",     "Delhi",     55000, 29),
    ("Divya",  "Engineering", "Pune",      91000, 33),
    ("Nitin",  "HR",          "Mumbai",    44000, 31),
    ("Anjali", "Finance",     "Bangalore", 67000, 28),
    ("Rahul",  "Engineering", "Delhi",     68000, 30),
    ("Pooja",  "HR",          "Pune",      41000, 26),
    ("Amit",   "Finance",     "Pune",      55000, 34),  # same salary as Karan
]

cols = ["name", "dept", "city", "salary", "age"]
df = spark.createDataFrame(data, cols)
display(df)

name,dept,city,salary,age
Ravi,Engineering,Pune,55000,28
Priya,HR,Mumbai,42000,32
Arjun,Engineering,Delhi,72000,26
Sneha,Finance,Pune,61000,30
Rohit,Engineering,Mumbai,80000,35
Meera,HR,Bangalore,39000,27
Karan,Finance,Delhi,55000,29
Divya,Engineering,Pune,91000,33
Nitin,HR,Mumbai,44000,31
Anjali,Finance,Bangalore,67000,28


In [0]:
# row_number — always unique sequential number
# even if two salaries are same — one gets 1, other gets 2
# order of tie-breaking is arbitrary

windowSpec = Window \
    .partitionBy("dept") \
    .orderBy(col("salary").desc())

df = df.withColumn("row_num", row_number().over(windowSpec))

display(df.orderBy("dept", "row_num"))

# Notice: Karan and Amit both 55000 in Finance
# row_number gives them 3 and 4 (no tie — arbitrary order)

name,dept,city,salary,age,row_num
Divya,Engineering,Pune,91000,33,1
Rohit,Engineering,Mumbai,80000,35,2
Arjun,Engineering,Delhi,72000,26,3
Rahul,Engineering,Delhi,68000,30,4
Ravi,Engineering,Pune,55000,28,5
Anjali,Finance,Bangalore,67000,28,1
Sneha,Finance,Pune,61000,30,2
Karan,Finance,Delhi,55000,29,3
Amit,Finance,Pune,55000,34,4
Nitin,HR,Mumbai,44000,31,1


In [0]:
# rank — ties get SAME rank, next rank SKIPS

windowSpec = Window \
    .partitionBy("dept") \
    .orderBy(col("salary").desc())

df = df.withColumn("sal_rank", rank().over(windowSpec))

display(df.orderBy("dept", "sal_rank"))

# Notice: Karan and Amit both 55000 in Finance
# Both get rank 3
# Next rank jumps to 5 (skips 4) ← this is rank() behaviour

name,dept,city,salary,age,row_num,sal_rank
Divya,Engineering,Pune,91000,33,1,1
Rohit,Engineering,Mumbai,80000,35,2,2
Arjun,Engineering,Delhi,72000,26,3,3
Rahul,Engineering,Delhi,68000,30,4,4
Ravi,Engineering,Pune,55000,28,5,5
Anjali,Finance,Bangalore,67000,28,1,1
Sneha,Finance,Pune,61000,30,2,2
Karan,Finance,Delhi,55000,29,3,3
Amit,Finance,Pune,55000,34,4,3
Nitin,HR,Mumbai,44000,31,1,1


In [0]:
# dense_rank — ties get SAME rank, next rank does NOT skip

windowSpec = Window \
    .partitionBy("dept") \
    .orderBy(col("salary").desc())

df = df.withColumn("dense_sal_rank", dense_rank().over(windowSpec))

display(df.orderBy("dept", "dense_sal_rank"))

# Notice: Karan and Amit both get dense_rank = 3
# Next rank is 4 (not 5) ← no gap, hence "dense"

name,dept,city,salary,age,row_num,sal_rank,dense_sal_rank
Divya,Engineering,Pune,91000,33,1,1,1
Rohit,Engineering,Mumbai,80000,35,2,2,2
Arjun,Engineering,Delhi,72000,26,3,3,3
Rahul,Engineering,Delhi,68000,30,4,4,4
Ravi,Engineering,Pune,55000,28,5,5,5
Anjali,Finance,Bangalore,67000,28,1,1,1
Sneha,Finance,Pune,61000,30,2,2,2
Karan,Finance,Delhi,55000,29,3,3,3
Amit,Finance,Pune,55000,34,4,3,3
Nitin,HR,Mumbai,44000,31,1,1,1


In [0]:
# See all three side by side — best way to understand difference
windowSpec = Window \
    .partitionBy("dept") \
    .orderBy(col("salary").desc())

result = df.select(
    "name", "dept", "salary",
    row_number().over(windowSpec).alias("row_num"),
    rank().over(windowSpec).alias("rank"),
    dense_rank().over(windowSpec).alias("dense_rank")
).orderBy("dept", "salary", ascending=[True, False])

display(result)

# Focus on Finance dept — Karan and Amit both 55000:
# row_num  → 3 and 4  (always unique)
# rank     → 3 and 3, next is 5  (skips)
# dense_rank → 3 and 3, next is 4  (no skip)

name,dept,salary,row_num,rank,dense_rank
Divya,Engineering,91000,1,1,1
Rohit,Engineering,80000,2,2,2
Arjun,Engineering,72000,3,3,3
Rahul,Engineering,68000,4,4,4
Ravi,Engineering,55000,5,5,5
Anjali,Finance,67000,1,1,1
Sneha,Finance,61000,2,2,2
Karan,Finance,55000,3,3,3
Amit,Finance,55000,4,3,3
Nitin,HR,44000,1,1,1


In [0]:

# ntile(4) divides employees into 4 equal salary buckets
# Bucket 1 = bottom 25%, Bucket 4 = top 25%
# Used for percentile grouping, bonus tiers

windowSpec = Window.orderBy(col("salary").desc())

df_tiles = df.withColumn("quartile", ntile(4).over(windowSpec))
display(df_tiles.orderBy("quartile", col("salary").desc()))

# Partition by dept for dept-level buckets
windowSpecDept = Window \
    .partitionBy("dept") \
    .orderBy(col("salary").desc())

df.withColumn("dept_quartile",
    ntile(4).over(windowSpecDept)).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


name,dept,city,salary,age,row_num,sal_rank,dense_sal_rank,quartile
Divya,Engineering,Pune,91000,33,1,1,1,1
Rohit,Engineering,Mumbai,80000,35,2,2,2,1
Arjun,Engineering,Delhi,72000,26,3,3,3,1
Rahul,Engineering,Delhi,68000,30,4,4,4,1
Anjali,Finance,Bangalore,67000,28,1,1,1,2
Sneha,Finance,Pune,61000,30,2,2,2,2
Ravi,Engineering,Pune,55000,28,5,5,5,2
Karan,Finance,Delhi,55000,29,3,3,3,3
Amit,Finance,Pune,55000,34,4,3,3,3
Nitin,HR,Mumbai,44000,31,1,1,1,3


name,dept,city,salary,age,row_num,sal_rank,dense_sal_rank,dept_quartile
Divya,Engineering,Pune,91000,33,1,1,1,1
Rohit,Engineering,Mumbai,80000,35,2,2,2,1
Arjun,Engineering,Delhi,72000,26,3,3,3,2
Rahul,Engineering,Delhi,68000,30,4,4,4,3
Ravi,Engineering,Pune,55000,28,5,5,5,4
Anjali,Finance,Bangalore,67000,28,1,1,1,1
Sneha,Finance,Pune,61000,30,2,2,2,2
Karan,Finance,Delhi,55000,29,3,3,3,3
Amit,Finance,Pune,55000,34,4,3,3,4
Nitin,HR,Mumbai,44000,31,1,1,1,1


In [0]:
# No partitionBy = rank across ALL employees globally

globalWindow = Window.orderBy(col("salary").desc())

result = df.withColumn("global_rank",
    dense_rank().over(globalWindow)) \
    .select("name", "dept", "salary", "global_rank") \
    .orderBy("global_rank")

display(result)
# Who is the highest paid overall?

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


name,dept,salary,global_rank
Divya,Engineering,91000,1
Rohit,Engineering,80000,2
Arjun,Engineering,72000,3
Rahul,Engineering,68000,4
Anjali,Finance,67000,5
Sneha,Finance,61000,6
Ravi,Engineering,55000,7
Karan,Finance,55000,7
Amit,Finance,55000,7
Nitin,HR,44000,8


In [0]:
# Production pattern — save ranked data
windowSpec = Window \
    .partitionBy("dept") \
    .orderBy(col("salary").desc())

df_final = df.withColumn("dept_rank",
    dense_rank().over(windowSpec))

spark.sql("DROP TABLE IF EXISTS employee_rankings")

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("employee_rankings")

print("✅ Saved!")
display(spark.table("employee_rankings"))

✅ Saved!


name,dept,city,salary,age,row_num,sal_rank,dense_sal_rank,dept_rank
Divya,Engineering,Pune,91000,33,1,1,1,1
Rohit,Engineering,Mumbai,80000,35,2,2,2,2
Arjun,Engineering,Delhi,72000,26,3,3,3,3
Rahul,Engineering,Delhi,68000,30,4,4,4,4
Ravi,Engineering,Pune,55000,28,5,5,5,5
Anjali,Finance,Bangalore,67000,28,1,1,1,1
Sneha,Finance,Pune,61000,30,2,2,2,2
Karan,Finance,Delhi,55000,29,3,3,3,3
Amit,Finance,Pune,55000,34,4,3,3,3
Nitin,HR,Mumbai,44000,31,1,1,1,1
